In [1]:
from pathlib import Path
import sys

# Find the DocuMind project directory
project_dir = Path.cwd()

# If the notebook is running inside notebooks/, go up one level
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

backend_dir = project_dir / "backend"

# Make the backend directory visible to Python
sys.path.insert(0, str(backend_dir))

print("Project directory:", project_dir)
print("Backend directory:", backend_dir)
print("Backend exists:", backend_dir.exists())

Project directory: /run/media/biplob/e1a9a613-7e49-4b9e-81ed-a1c7ad2b4cae/biplob/DocuMind
Backend directory: /run/media/biplob/e1a9a613-7e49-4b9e-81ed-a1c7ad2b4cae/biplob/DocuMind/backend
Backend exists: True


In [2]:
%pip install PyMuPDF

Note: you may need to restart the kernel to use updated packages.


In [3]:
from app.services.text_splitter import (
    split_text,
    search_chunks,
    search_index,
    build_index,
    chunk_pages,
)
from app.services.document_loader import extract_pages

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All imports successful!")

All imports successful!


In [4]:
pages = extract_pages(Path("../data/sample.pdf").read_bytes())
chunks = chunk_pages(pages, chunk_size=500, overlap=50)
texts = [c["text"] for c in chunks]
print("pages:", len(pages), "| chunks:", len(chunks))
print(chunks[0])

pages: 3 | chunks: 7
{'page': 1, 'text': 'Python is a programming language known for readable syntax. Python is a programming\nlanguage known for readable syntax. Python is a programming language known for readable\nsyntax. Python is a programming language known for readable syntax. Python is a\nprogramming language known for readable syntax. Python is a programming language known\nfor readable syntax. Python is a programming language known for readable syntax. Python is\na programming language known for readable syntax. Python is a'}


In [5]:
vectorizer, matrix = build_index(texts)


def show(query, top_k=3, min_score=0.0):
    hits = search_index(query, vectorizer, matrix, top_k, min_score)
    print(f"\nQuestion: {query}")
    if not hits:
        print(" (no match)")
    for i, score in hits:
        c = chunks[i]
        print(
            f" page {c['page']} | chunk {i:>2} | score {score:.3f} | {c['text'][:120]!r}"
        )


show("What is FastAPI used for?")


Question: What is FastAPI used for?
 page 3 | chunk  5 | score 0.447 | 'FastAPI is a web framework for building APIs with Python. FastAPI is a web framework for\nbuilding APIs with Python. Fast'
 page 3 | chunk  6 | score 0.407 | 'APIs with\nPython. FastAPI is a web framework for building APIs with Python. FastAPI is a web\nframework for building APIs'


In [6]:
tests = [
    ("What is FastAPI used for?", "web framework"),
    ("Which database stores data in tables?", "relational database"),
    ("What is Python known for?", "readable syntax"),
    ("How do I build an API?", "web framework"),
]


def rank(query, texts, **opts):
    vec = TfidfVectorizer(**opts)
    matrix = vec.fit_transform(texts)
    scores = cosine_similarity(vec.transform([query]), matrix).flatten()
    return scores.argsort()[::-1], scores


def evaluate(texts, tests, top_k=3, verbose=False, **opts):
    top1 = found = 0
    for question, expected in tests:
        order, scores = rank(question, texts, **opts)
        position = None
        for place, idx in enumerate(order[:top_k], start=1):
            if scores[idx] <= 0:
                break
            if expected.lower() in texts[idx].lower():
                position = place
                break
        top1 += position == 1
        found += position is not None
        if verbose:
            print(f"{position or 'miss':>5} | best score {scores[order[0]]:.3f} | {question}")
    return top1, found


top1, found = evaluate(texts, tests, verbose=True, stop_words="english")
print(f"\ncorrect chunk first: {top1}/{len(tests)} | within top 3: {found}/{len(tests)}")

    1 | best score 0.447 | What is FastAPI used for?
    1 | best score 0.816 | Which database stores data in tables?
    1 | best score 0.545 | What is Python known for?
 miss | best score 0.000 | How do I build an API?

correct chunk first: 3/4 | within top 3: 3/4


In [7]:
for size, overlap in [(300, 30), (500, 50), (800, 80), (1200, 120)]:
    c = [x["text"] for x in chunk_pages(pages, size, overlap)]
    top1, found = evaluate(c, tests, stop_words="english")
    print(
        f"size {size:>4} overlap {overlap:>3} | chunks {len(c):>3} | first {top1}/{len(tests)} | top3 {found}/{len(tests)}"
    )

size  300 overlap  30 | chunks  12 | first 3/4 | top3 3/4
size  500 overlap  50 | chunks   7 | first 3/4 | top3 3/4
size  800 overlap  80 | chunks   6 | first 3/4 | top3 3/4
size 1200 overlap 120 | chunks   3 | first 3/4 | top3 3/4


In [8]:
vec = TfidfVectorizer(stop_words="english").fit(texts)
terms = vec.build_analyzer()("How do I build an API?")
print(terms, [t for t in terms if t in vec.vocabulary_])
for i, t in enumerate(texts):
    if any(x in vec.build_analyzer()(t) for x in terms):
        print(i, repr(t[-40:]), "| next:", repr(texts[i + 1][:40]) if i + 1 < len(texts) else "")

['build', 'api'] []
